In [1]:
from typing import List
import hashlib
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import torch
import pandas as pd



device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_function = SentenceTransformerEmbeddingFunction(
    model_name="dangvantuan/vietnamese-embedding",
    device=device,
    normalize_embeddings=True,
)

def make_id(text: str) -> str:
    return hashlib.md5(text.encode()).hexdigest()

chroma_client = chromadb.PersistentClient(path="./vectordb")

C:\Users\Admin\Desktop\HK252\DATN\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3833.00it/s]


In [2]:
policy_collection = chroma_client.get_or_create_collection(
    name="policies",
    embedding_function=embedding_function,
)



POLICY_TEXTS: List[str] = [
    """
Tất cả sản phẩm là hàng chính hãng và được bảo hành theo tiêu chuẩn nhà sản xuất.

Có các gói bảo hành mở rộng giúp tăng trải nghiệm sử dụng.
""",
    """
Gói 1 đổi 1 VIP:
- Đổi mới nếu lỗi phần cứng do nhà sản xuất
- Áp dụng cho điện thoại, tablet, tai nghe, smartwatch
- Không áp dụng cho máy móp méo, vào nước
- Thời gian xử lý: 24h đến 14 ngày
""",
    """
Gói rơi vỡ - vào nước:
- Hỗ trợ phần lớn chi phí sửa chữa
- Không sửa được sẽ đổi thiết bị tương đương
- Có phí dịch vụ khi đổi
""",
    """
Gói bảo hành mở rộng cao cấp:
- Gia hạn sau khi hãng hết bảo hành
- Miễn phí sửa lỗi nhà sản xuất
- Đổi máy nếu hư hỏng nặng
""",
    """
Chính sách giao hàng:
- Nội thành: 1-2 giờ hoặc trong ngày
- Ngoại thành: 24-48 giờ
- Liên tỉnh: 2-5 ngày
- Có thể miễn phí vận chuyển tùy thành viên
""",
    """
Chính sách thanh toán:
- Thanh toán khi nhận hàng
- Chuyển khoản ngân hàng
- Ví điện tử, QR code
- Trả góp qua thẻ tín dụng
""",
    """
Chính sách hoàn tiền:
- Tiền mặt: hoàn ngay
- Chuyển khoản: 1-3 ngày
- Thẻ: 7-15 ngày
- Ví điện tử: 3-8 ngày
""",
    """
Chính sách đổi trả:
- Đổi trong 30-35 ngày
- Sản phẩm còn nguyên trạng
- Không áp dụng nếu lỗi do người dùng
""",
    """
Chính sách dữ liệu:
- Người dùng tự sao lưu
- Không chịu trách nhiệm mất dữ liệu
- Có thể hỗ trợ chuyển dữ liệu nếu có đồng ý
""",
    """
Chính sách kiểm tra sản phẩm:
- Thanh toán 100% trước khi mở hộp
- Kiểm tra ngoại hình trước khi kích hoạt
- Lỗi thẩm mỹ: đổi sản phẩm
"""
]



policy_ids = [make_id(t) for t in POLICY_TEXTS]    
policy_collection.add(
    ids=policy_ids,
    documents=POLICY_TEXTS,
)

print(f"Inserted {len(POLICY_TEXTS)} policy documents into Chroma")

Inserted 10 policy documents into Chroma


In [3]:
product_collection = chroma_client.get_or_create_collection(
    name="products",
    embedding_function=embedding_function,
)



def parse_faq(faq_str):
    if pd.isna(faq_str) or faq_str.strip() == "":
        return []
    faqs = faq_str.split("### Câu hỏi:")
    qa_list = []
    for faq in faqs[1:]:
        parts = faq.split("### Câu trả lời:")
        if len(parts) == 2:
            question = parts[0].strip()
            answer = parts[1].strip()
            qa_list.append(f"{question}\n{answer}")
    return qa_list



def process_row(row, chroma_collection):
    chunks = []
    
    desc = row.get("description")
    if pd.notna(desc) and str(desc).strip():
        chunks.append(str(desc).strip())

    faq = row.get("faq")
    if pd.notna(faq) and str(faq).strip():
        faqs = parse_faq(faq)
        for f in faqs:
            chunks.append(f.strip())

    if not chunks:
        return
            
    docs = []

    for chunk in chunks:
        text = chunk.replace("\n\n---", "").strip()
        if not text:
            print("lỗi clean faq")
            return

        docs.append(text)

    if not docs:
        print("lỗi không có doc")
        return

    product_ids = [make_id(d) for d in docs]    
    chroma_collection.add(
        ids=product_ids,
        documents=docs,
    )





df_laptop = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/laptop.csv")
df_man_hinh = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/man-hinh.csv")
df_may_in = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/may-in.csv")
df_micro = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/micro-thu-am.csv")
df_mobile = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/mobile.csv")
df_tablet = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/tablet.csv")
df_tivi = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/tivi.csv")
df = pd.concat([df_laptop, df_man_hinh, df_may_in, df_micro, df_mobile, df_tablet, df_tivi], ignore_index=True)





for idx, row in df.iterrows():
    process_row(row, product_collection)
print(f"Inserted {len(df)} product documents into Chroma")

Inserted 975 product documents into Chroma
